# ch9. 기저·차원·계수(rank) — 진짜 정보는 몇 차원인가 🧮

웹 챕터 9에서 배운 내용을 NumPy 로 직접 계산해 봅니다.

**이 노트북에서 하는 것**
1. 선형결합을 잔뜩 만들어 **span(생성)** 을 그림으로 확인: 독립이면 평면, 종속이면 직선
2. **행렬식**과 `np.linalg.matrix_rank` 로 독립/종속 판별
3. 중복 열이 있는 데이터 행렬의 **rank = 진짜 정보량** 재기 (키 cm / 키 inch 예제)
4. AI 미리보기: **rank 1 짜리 평점 행렬** (ch11 SVD 예고)

> 용어 복습. **선형결합**: 벡터들에 숫자(계수)를 곱해 더한 것. **span**: 선형결합으로 갈 수 있는 모든 곳.
> **선형독립**: 전부 새 방향(정보). **rank**: 행렬에서 선형독립인 열의 개수 = 진짜 정보의 차원.

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch9')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (그래프의 한글이 깨지지 않게)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('numpy', np.__version__, '준비 완료')

## 1. span 을 눈으로: 계수를 바꿔가며 선형결합을 잔뜩 찍어보기

두 벡터 $\vec{u}, \vec{v}$ 의 선형결합 $\vec{w} = c_1\vec{u} + c_2\vec{v}$ 에서
계수 $c_1, c_2$ 를 골고루 바꿔가며 $\vec{w}$ 를 수백 개 만들어 점으로 찍으면,
그 점들이 채우는 영역이 바로 **span** 입니다.

In [ ]:
def plot_span(u, v, title):
    """계수 c1, c2 를 -3~3 으로 바꿔가며 선형결합 w = c1*u + c2*v 를 점으로 찍는다."""
    cs = np.linspace(-3, 3, 25)
    points = np.array([c1 * u + c2 * v for c1 in cs for c2 in cs])

    plt.figure(figsize=(5, 5))
    plt.scatter(points[:, 0], points[:, 1], s=4, color='lightgray', label='선형결합 w 들 (= span)')
    plt.quiver([0, 0], [0, 0], [u[0], v[0]], [u[1], v[1]],
               angles='xy', scale_units='xy', scale=1, color=['#6366f1', '#f59e0b'])
    plt.text(u[0], u[1], ' u', color='#6366f1', fontsize=12, fontweight='bold')
    plt.text(v[0], v[1], ' v', color='#f59e0b', fontsize=12, fontweight='bold')
    plt.axhline(0, color='gray', lw=0.5)
    plt.axvline(0, color='gray', lw=0.5)
    plt.xlim(-9, 9); plt.ylim(-9, 9)
    plt.gca().set_aspect('equal')
    plt.title(title)
    plt.legend(loc='upper left')
    plt.show()

# 독립인 두 벡터 — 점들이 평면을 빈틈없이 채운다 (2차원어치 정보)
plot_span(np.array([2, 1]), np.array([-1, 2]), '독립인 u, v: span 이 평면을 채운다')

### 종속이면 어떻게 될까?

이번엔 $\vec{v} = 2\vec{u}$ (평행 = **종속**) 으로 같은 실험을 합니다.
계수를 아무리 바꿔도 점들이 **한 직선을 벗어나지 못해요**. 재료가 사실상 한 방향뿐이라서요.

In [ ]:
# 종속인 두 벡터 (v = 2u) — 점들이 직선 하나에 갇힌다 (1차원어치 정보)
plot_span(np.array([2, 1]), np.array([4, 2]), '종속인 u, v (v=2u): span 이 직선에 갇힌다')

## 2. 독립/종속 판별: 행렬식과 `matrix_rank`

2차원 벡터 2개는 **행렬식**으로 판별했죠: 두 벡터를 열로 세운 행렬에서
$\det = u_x v_y - u_y v_x$ 가 **0이면 종속, 0이 아니면 독립**.

벡터가 많아지거나 차원이 높아지면 손 계산이 힘들어지는데,
NumPy 의 **`np.linalg.matrix_rank`** 가 독립인 열의 개수(= rank)를 바로 세 줍니다.

In [ ]:
def det2(u, v):
    return u[0] * v[1] - u[1] * v[0]

pairs = {
    'u=(2,1), v=(-1,2)': (np.array([2, 1]), np.array([-1, 2])),
    'u=(2,1), v=(4,2) ': (np.array([2, 1]), np.array([4, 2])),
}

for name, (u, v) in pairs.items():
    M = np.column_stack([u, v])          # 두 벡터를 열로 세운 2x2 행렬
    r = np.linalg.matrix_rank(M)
    verdict = '독립 (평면 전체 생성)' if r == 2 else '종속 (직선에 갇힘)'
    print(f'{name}: det = {det2(u, v):3d}, rank = {r} -> {verdict}')

## 3. 데이터 행렬의 rank = 진짜 정보량

웹 챕터의 예제 그대로, 회원 8명의 **키(cm) · 키(inch) · 몸무게** 데이터를 만들어 봅니다.
키(inch)는 키(cm) × 0.394 일 뿐이라 **새 정보가 없어요**.
열은 3개지만 rank 는 몇으로 나올까요?

In [ ]:
rng = np.random.default_rng(0)
height_cm = rng.uniform(150, 190, size=8)
height_in = height_cm * 0.394          # 키(cm)의 스칼라배 — 완전히 중복인 열
weight = rng.uniform(45, 90, size=8)

X = np.column_stack([height_cm, height_in, weight])
print('데이터 행렬 X (열: 키cm, 키inch, 몸무게)')
print(X.round(1))
print()
print('열 개수 :', X.shape[1])
print('rank    :', np.linalg.matrix_rank(X), ' <- 진짜 정보는 2차원어치!')

겉보기 열은 3개인데 **rank 는 2**입니다. "키" 정보 하나와 "몸무게" 정보 하나뿐이라는 걸
컴퓨터가 스스로 알아낸 겁니다. 실무에서는 이런 중복(다중공선성)을 찾아 정리해야
모델 학습이 안정적이에요.

## 4. AI 미리보기: rank 1 짜리 평점 행렬 (ch11 SVD 예고)

추천 시스템의 평점 행렬은 거대해 보여도 실제로는 **"취향 축" 몇 개짜리 낮은 rank** 에 가깝습니다.
가장 극단적인 장난감 예제로, 세상에 **액션 취향 축 하나만** 있다고 해봅시다.

$$\text{평점}(사용자 i, 영화 j) = \text{취향}_i \times \text{액션 농도}_j$$

In [ ]:
taste = np.array([1.0, 0.5, 0.0, 0.9])          # 사용자 4명의 액션 취향
action = np.array([5.0, 4.0, 1.0, 2.0, 4.5])    # 영화 5편의 액션 농도

R = np.outer(taste, action)                      # 4x5 평점 행렬 = 취향 x 농도
print('평점 행렬 R (행: 사용자, 열: 영화)')
print(R.round(2))
print()
print('모양 :', R.shape, '(숫자 20개)')
print('rank :', np.linalg.matrix_rank(R), ' <- 취향 축 1개면 전부 설명된다!')
print('실제 필요한 숫자 :', taste.size + action.size, '개 (취향 4 + 농도 5)')

20개짜리 행렬이 숫자 9개로 완전히 표현됐어요. 실제 데이터는 rank 가 딱 1은 아니지만
**"낮은 rank + 약간의 잡음"** 인 경우가 많아서, 낮은 rank 로 근사하면 저장·계산이 확 줄고
잡음까지 걸러집니다. 이 **저랭크 근사**를 자동으로 해 주는 도구가 ch11 의 **SVD** 예요.

---

### 정리

| 개념 | 한 줄 요약 | NumPy |
|------|-----------|-------|
| 선형결합 | 벡터를 늘리고 더해 섞기 | `c1*u + c2*v` |
| span | 선형결합으로 갈 수 있는 모든 곳 | (점 찍어 확인) |
| 독립/종속 | 전부 새 정보인가 / 재탕이 있나 | `matrix_rank` 로 판별 |
| 기저·차원 | 최소 재료 세트 · 그 개수 | — |
| **rank** | **진짜 정보의 차원** | `np.linalg.matrix_rank(A)` |

**다음 챕터(ch10)**: 직교성과 투영. "가장 가까운 답"을 찾는 최소제곱의 원리입니다.